In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import mlflow
from sklearn.metrics import roc_auc_score, average_precision_score, brier_score_loss

# Set up MLflow: store runs in mlruns/ at the project root
mlflow.set_tracking_uri("file:../mlruns")
mlflow.set_experiment("loan-default")

print(f"MLflow tracking URI: {mlflow.get_tracking_uri()}")
print(f"Experiment: {mlflow.get_experiment_by_name('loan-default').name}")

In [ ]:
df = pd.read_parquet('../data/loans_clean.parquet')

# Apply EDA cleaning
df['dti'] = df['dti'].replace([-1, 999], np.nan).clip(upper=100)
df['credit_history_months'] = df['credit_history_months'].replace(999, np.nan)
df['annual_inc'] = df['annual_inc'].clip(upper=1_000_000)
df['revol_util'] = df['revol_util'].clip(upper=100)

# Temporal split
df_train = df[df['issue_year'].isin([2014, 2015])].copy()
df_val   = df[df['issue_year'] == 2016].copy()
df_test  = df[df['issue_year'] == 2017].copy()

y_train = df_train['default'].values
y_val   = df_val['default'].values

print(f"Train: {len(df_train):,}  |  Val: {len(df_val):,}  |  Test: {len(df_test):,}")

In [ ]:
GAIN_RATE = 0.10
LOSS_RATE = 0.50

def calculate_profit(df_subset, approval_mask):
    approved = df_subset[approval_mask]
    if len(approved) == 0:
        return 0
    profit = np.where(
        approved['default'] == 0,
         GAIN_RATE * approved['loan_amnt'],
        -LOSS_RATE * approved['loan_amnt']
    )
    return profit.sum()

In [ ]:
with mlflow.start_run(run_name="baseline_approve_all"):
    # Parameters describing the strategy
    mlflow.log_param("model_type", "constant")
    mlflow.log_param("predicted_default_prob", float(df_train['default'].mean()))
    
    # Predictions and metrics on val
    probs = np.full(len(df_val), df_train['default'].mean())
    profit = calculate_profit(df_val, np.ones(len(df_val), dtype=bool))
    
    mlflow.log_metric("auc", 0.5)
    mlflow.log_metric("pr_auc", float(y_val.mean()))  # PR-AUC of random is base rate
    mlflow.log_metric("brier", float(brier_score_loss(y_val, probs)))
    mlflow.log_metric("profit_at_threshold", float(profit))
    mlflow.log_metric("approval_rate", 1.0)
    
    print(f"Logged baseline_approve_all")

In [ ]:
with mlflow.start_run(run_name="baseline_grade_only"):
    # Learn from train
    grade_rates = df_train.groupby('grade')['default'].mean()
    val_probs = df_val['grade'].map(grade_rates).values
    
    # Find optimal threshold (same logic as before)
    thresholds = np.linspace(0.05, 0.55, 100)
    profits = [calculate_profit(df_val, val_probs < t) for t in thresholds]
    best_idx = int(np.argmax(profits))
    best_threshold = float(thresholds[best_idx])
    best_profit = float(profits[best_idx])
    approval_rate = float((val_probs < best_threshold).mean())
    
    # Log
    mlflow.log_param("model_type", "grade_lookup")
    mlflow.log_param("best_threshold", best_threshold)
    
    mlflow.log_metric("auc", float(roc_auc_score(y_val, val_probs)))
    mlflow.log_metric("pr_auc", float(average_precision_score(y_val, val_probs)))
    mlflow.log_metric("brier", float(brier_score_loss(y_val, val_probs)))
    mlflow.log_metric("profit_at_threshold", best_profit)
    mlflow.log_metric("approval_rate", approval_rate)
    
    print(f"Logged baseline_grade_only — AUC: {roc_auc_score(y_val, val_probs):.4f}, Profit: ${best_profit:,.0f}")

In [ ]:
# Make a copy we can transform without touching the original
df_train_fe = df_train.copy()
df_val_fe = df_val.copy()
df_test_fe = df_test.copy()


def prep_features(df):
    """Apply the feature engineering decisions from EDA."""
    df = df.copy()
    
    # 1. Drop redundant / leaky-correlation features (from EDA Section 5.4)
    redundant = [
        'fico_range_high',       # 1.000 correlated with fico_range_low
        'funded_amnt',           # essentially loan_amnt
        'funded_amnt_inv',       # essentially loan_amnt
        'num_sats',              # essentially open_acc
        'installment',           # derived from loan_amnt + term + int_rate
        'num_rev_tl_bal_gt_0',   # essentially num_actv_rev_tl
    ]
    
    # 2. Drop joint/sec_app features (98% missing — only for joint apps)
    joint_cols = [c for c in df.columns if c.startswith('sec_app_') or c.endswith('_joint')]
    
    # 3. Drop high-cardinality nuisance columns
    high_cardinality = ['zip_code', 'sub_grade']  # keep grade, drop sub_grade for now
    
    # 4. Drop the columns we used for splitting (don't feed to model)
    split_cols = ['issue_year']
    
    cols_to_drop = redundant + joint_cols + high_cardinality + split_cols
    df = df.drop(columns=[c for c in cols_to_drop if c in df.columns])
    
    # 5. Parse emp_length from string to integer
    #    "< 1 year" → 0, "1 year" → 1, ..., "10+ years" → 10, NaN → NaN
    emp_map = {
        '< 1 year': 0, '1 year': 1, '2 years': 2, '3 years': 3, '4 years': 4,
        '5 years': 5, '6 years': 6, '7 years': 7, '8 years': 8, '9 years': 9,
        '10+ years': 10
    }
    df['emp_length'] = df['emp_length'].map(emp_map)
    
    return df


df_train_fe = prep_features(df_train_fe)
df_val_fe = prep_features(df_val_fe)
df_test_fe = prep_features(df_test_fe)

print(f"Features after prep: {df_train_fe.shape[1]}")
print(f"\nTrain shape: {df_train_fe.shape}")
print(f"Val shape:   {df_val_fe.shape}")

In [ ]:
# Separate target from features
y_train = df_train_fe['default'].values
y_val = df_val_fe['default'].values

X_train = df_train_fe.drop(columns=['default'])
X_val = df_val_fe.drop(columns=['default'])

# Parse `term` from string ("36 months", "60 months") to integer (36, 60)
X_train['term'] = X_train['term'].str.extract(r'(\d+)').astype(int)
X_val['term']   = X_val['term'].str.extract(r'(\d+)').astype(int)

# Identify categorical vs numeric columns
categorical_cols = X_train.select_dtypes(include=['object']).columns.tolist()
numeric_cols = X_train.select_dtypes(include=[np.number]).columns.tolist()

print(f"Categorical columns ({len(categorical_cols)}): {categorical_cols}")
print(f"\nNumeric columns: {len(numeric_cols)} features")

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression

# Numeric pipeline: median impute → standardize
numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

# Categorical pipeline: fill missing with "missing" string → one-hot encode
categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='constant', fill_value='missing')),
    ('encoder', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

# Combine into one preprocessor
preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_cols),
        ('cat', categorical_transformer, categorical_cols)
    ]
)

# Wrap with logistic regression
clf = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', LogisticRegression(
        class_weight='balanced',
        max_iter=1000,
        solver='lbfgs',
        random_state=42,
        n_jobs=-1
    ))
])

print("Pipeline built. Steps:")
for name, step in clf.named_steps.items():
    print(f"  {name}: {type(step).__name__}")

In [ ]:
import time

with mlflow.start_run(run_name="logreg_baseline"):
    # Log parameters
    mlflow.log_param("model_type", "LogisticRegression")
    mlflow.log_param("class_weight", "balanced")
    mlflow.log_param("max_iter", 1000)
    mlflow.log_param("solver", "lbfgs")
    mlflow.log_param("n_features_input", X_train.shape[1])
    
    # Train
    start = time.time()
    clf.fit(X_train, y_train)
    train_time = time.time() - start
    
    # Predict probabilities on val
    val_probs = clf.predict_proba(X_val)[:, 1]
    
    # Core metrics
    auc = roc_auc_score(y_val, val_probs)
    pr_auc = average_precision_score(y_val, val_probs)
    brier = brier_score_loss(y_val, val_probs)
    
    # Optimal threshold for profit
    thresholds = np.linspace(0.05, 0.55, 100)
    profits = [calculate_profit(df_val, val_probs < t) for t in thresholds]
    best_idx = int(np.argmax(profits))
    best_threshold = float(thresholds[best_idx])
    best_profit = float(profits[best_idx])
    approval_rate = float((val_probs < best_threshold).mean())
    
    # Log
    mlflow.log_param("best_threshold", best_threshold)
    mlflow.log_metric("auc", auc)
    mlflow.log_metric("pr_auc", pr_auc)
    mlflow.log_metric("brier", brier)
    mlflow.log_metric("profit_at_threshold", best_profit)
    mlflow.log_metric("approval_rate", approval_rate)
    mlflow.log_metric("train_time_seconds", train_time)
    
    # Log the trained model as an artifact
    mlflow.sklearn.log_model(clf, "model")
    
    print(f"=== LOGISTIC REGRESSION ===")
    print(f"Train time:     {train_time:.1f}s")
    print(f"AUC:            {auc:.4f}")
    print(f"PR-AUC:         {pr_auc:.4f}")
    print(f"Brier:          {brier:.4f}")
    print(f"Best threshold: {best_threshold:.4f}")
    print(f"Approval rate:  {approval_rate:.2%}")
    print(f"Profit:         ${best_profit:,.0f}")
    
    print(f"\n--- Comparison to baselines ---")
    print(f"Grade-only AUC:    0.6816  →  LogReg:  {auc:.4f}  (delta: {auc - 0.6816:+.4f})")
    print(f"Grade-only profit: $42.8M  →  LogReg:  ${best_profit/1e6:.1f}M  (delta: ${(best_profit - 42_848_472)/1e6:+.1f}M)")

In [ ]:
# Drop int_rate from features and column list
X_train_no_rate = X_train.drop(columns=['int_rate'])
X_val_no_rate   = X_val.drop(columns=['int_rate'])
numeric_cols_no_rate = [c for c in numeric_cols if c != 'int_rate']

# Rebuild the preprocessor with the reduced numeric list
preprocessor_no_rate = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_cols_no_rate),
        ('cat', categorical_transformer, categorical_cols)
    ]
)

clf_no_rate = Pipeline(steps=[
    ('preprocessor', preprocessor_no_rate),
    ('classifier', LogisticRegression(
        class_weight='balanced',
        max_iter=1000,
        solver='lbfgs',
        random_state=42,
        n_jobs=-1
    ))
])

with mlflow.start_run(run_name="logreg_no_int_rate"):
    mlflow.log_param("model_type", "LogisticRegression")
    mlflow.log_param("class_weight", "balanced")
    mlflow.log_param("max_iter", 1000)
    mlflow.log_param("excluded_feature", "int_rate")
    mlflow.log_param("n_features_input", X_train_no_rate.shape[1])
    
    start = time.time()
    clf_no_rate.fit(X_train_no_rate, y_train)
    train_time = time.time() - start
    
    val_probs = clf_no_rate.predict_proba(X_val_no_rate)[:, 1]
    
    auc = roc_auc_score(y_val, val_probs)
    pr_auc = average_precision_score(y_val, val_probs)
    brier = brier_score_loss(y_val, val_probs)
    
    thresholds = np.linspace(0.05, 0.55, 100)
    profits = [calculate_profit(df_val, val_probs < t) for t in thresholds]
    best_idx = int(np.argmax(profits))
    best_threshold = float(thresholds[best_idx])
    best_profit = float(profits[best_idx])
    approval_rate = float((val_probs < best_threshold).mean())
    
    mlflow.log_param("best_threshold", best_threshold)
    mlflow.log_metric("auc", auc)
    mlflow.log_metric("pr_auc", pr_auc)
    mlflow.log_metric("brier", brier)
    mlflow.log_metric("profit_at_threshold", best_profit)
    mlflow.log_metric("approval_rate", approval_rate)
    mlflow.log_metric("train_time_seconds", train_time)
    
    mlflow.sklearn.log_model(clf_no_rate, "model")
    
    print(f"=== LOGISTIC REGRESSION (no int_rate) ===")
    print(f"Train time:     {train_time:.1f}s")
    print(f"AUC:            {auc:.4f}")
    print(f"PR-AUC:         {pr_auc:.4f}")
    print(f"Brier:          {brier:.4f}")
    print(f"Best threshold: {best_threshold:.4f}")
    print(f"Approval rate:  {approval_rate:.2%}")
    print(f"Profit:         ${best_profit:,.0f}")
    
    print(f"\n--- Comparison: with vs without int_rate ---")
    print(f"With int_rate:    AUC 0.7157, Profit $60.7M")
    print(f"Without int_rate: AUC {auc:.4f}, Profit ${best_profit/1e6:.1f}M")
    print(f"Cost of dropping: AUC {auc - 0.7157:+.4f}, Profit ${(best_profit - 60_676_972)/1e6:+.1f}M")